In [3]:
from pynq import Overlay, MMIO, allocate
import ipywidgets as widgets
from IPython.display import display
import cv2, time
import numpy as np

print("1. Loading FPGA overlay...")
overlay = Overlay("main_flow.bit")

width, height = 1280, 720
# Corrected to 3 bytes per pixel
frame_buffer = allocate(shape=(height, width, 3), dtype=np.uint8)

S2MM_CR, S2MM_SR = 0x30, 0x34
S2MM_VSIZE, S2MM_HSIZE, S2MM_STRIDE = 0xA0, 0xA4, 0xA8
S2MM_ADDR1, S2MM_ADDR2, S2MM_ADDR3 = 0xAC, 0xB0, 0xB4

info = overlay.ip_dict['axi_vdma_0']
vdma = MMIO(info['phys_addr'], info['addr_range'])

try:
    # Soft reset
    vdma.write(S2MM_CR, 0x4)
    while vdma.read(S2MM_CR) & 0x4:
        pass
    vdma.write(S2MM_SR, 0xFFFFFFFF) 

    # Populate all 3 addresses
    for off in (S2MM_ADDR1, S2MM_ADDR2, S2MM_ADDR3):
        vdma.write(off, frame_buffer.physical_address)

    vdma.write(S2MM_HSIZE, width * 3)        
    vdma.write(S2MM_STRIDE, width * 3)

    vdma.write(S2MM_CR, 0x1)                 
    vdma.write(S2MM_VSIZE, height)           

    time.sleep(0.5)
    status = vdma.read(S2MM_SR)
    print(f"S2MM Status: {hex(status)}")

    ERR_MASK = 0xFF0
    if status & 0x1:
        print("VDMA halted")
    elif status & ERR_MASK:
        print(f"VDMA error flags: {hex(status & ERR_MASK)}")
    else:
        print("VDMA running, no errors — starting video...")
        image_widget = widgets.Image(format='jpeg', width=640, height=360)
        display(image_widget)
        
        while True:
            frame_buffer.invalidate()
            frame = cv2.cvtColor(frame_buffer, cv2.COLOR_RGB2BGR)
            _, jpeg = cv2.imencode('.jpg', frame)
            image_widget.value = jpeg.tobytes()
            time.sleep(0.03)

except KeyboardInterrupt:
    print("\nUser interrupted stream.")
except Exception as e:
    print(f"\nError occurred: {e}")
finally:
    print("Cleaning up hardware and memory...")
    vdma.write(S2MM_CR, 0x0)
    frame_buffer.freebuffer()
    print("Cleanup complete.")

1. Loading FPGA overlay...
VDMA running. Video started.



Cleaning up hardware and memory...
Cleanup complete. Safe to run again.


KeyboardInterrupt: 